# Linear Support Vector Machine Tutorial: Email Spam Detection

This notebook uses the real-life **UCI Spambase** dataset to classify emails as spam or non-spam with a **Linear Support Vector Machine**.

- Official dataset page: https://archive.ics.uci.edu/dataset/94/spambase
- Direct download: https://archive.ics.uci.edu/static/public/94/spambase.zip
- Original observations: 4,601 emails
- Predictors: 57 numeric word-frequency, character-frequency, and capital-run features
- Target: `spam` (`1` = spam and `0` = non-spam)

The tutorial includes preprocessing, exploratory visualization, `SVC(kernel='linear')`, hyperparameter tuning, evaluation metrics, and result visualization.

In [ ]:
# 1. IMPORT LIBRARIES
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)

sns.set_theme(style='whitegrid', palette='Set2')
RANDOM_STATE = 42

In [ ]:
# 2. LOAD THE REAL-LIFE DATASET FROM UCI
DATA_URL = 'https://archive.ics.uci.edu/static/public/94/spambase.zip'

with urlopen(DATA_URL) as response:
    zip_bytes = BytesIO(response.read())

with ZipFile(zip_bytes) as archive:
    # Extract feature names from the official UCI names file.
    names_text = archive.read('spambase.names').decode('utf-8')
    feature_names = [
        line.split(':', 1)[0].strip()
        for line in names_text.splitlines()
        if ':' in line and not line.strip().startswith('|')
    ]

    if len(feature_names) != 57:
        raise ValueError(f'Expected 57 feature names, found {len(feature_names)}')

    with archive.open('spambase.data') as data_file:
        df = pd.read_csv(data_file, header=None, names=feature_names + ['spam'])

print('Dataset shape:', df.shape)
display(df.head())

In [ ]:
# 3. INSPECT DATA QUALITY
print('Number of numeric columns:', df.select_dtypes(include=np.number).shape[1])
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
print('\nTarget counts:\n', df['spam'].value_counts().sort_index())
print('\nTarget proportions:\n', df['spam'].value_counts(normalize=True).sort_index().round(4))
display(df.describe().T.head(10))

In [ ]:
# 4. PREPROCESS THE DATA
# Remove exact duplicate rows before splitting to reduce train-test contamination.
df = df.drop_duplicates().reset_index(drop=True)

# Convert every column to numeric; invalid values become NaN.
df = df.apply(pd.to_numeric, errors='coerce')

# Separate predictors and target.
X = df.drop(columns=['spam'])
y = df['spam']

# Median imputation provides a robust response to any missing predictor values.
X = X.fillna(X.median())

# Rows with a missing target cannot be used for supervised learning.
valid_target = y.notna()
X = X.loc[valid_target].reset_index(drop=True)
y = y.loc[valid_target].astype(int).reset_index(drop=True)

print('Shape after preprocessing:', X.shape)
print('Remaining predictor missing values:', int(X.isna().sum().sum()))
print('Target labels:', sorted(y.unique()))

## Exploratory data visualization

The following plots inspect class balance, feature distributions, correlations with the target, and the approximate class structure in two principal components.

In [ ]:
# 5. VISUALIZE THE TARGET DISTRIBUTION
class_counts = y.map({0: 'Non-spam', 1: 'Spam'}).value_counts()

plt.figure(figsize=(6, 4))
ax = sns.barplot(
    x=class_counts.index, y=class_counts.values,
    hue=class_counts.index, legend=False
)
for container in ax.containers:
    ax.bar_label(container)
plt.title('Email Class Distribution')
plt.xlabel('Email Class')
plt.ylabel('Number of Emails')
plt.tight_layout()
plt.show()

In [ ]:
# 6. VISUALIZE REPRESENTATIVE FEATURE DISTRIBUTIONS
plot_df = X.copy()
plot_df['Email Class'] = y.map({0: 'Non-spam', 1: 'Spam'})
selected_features = [
    'word_freq_remove', 'word_freq_free',
    'char_freq_$', 'capital_run_length_total'
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for feature, ax in zip(selected_features, axes.ravel()):
    sns.boxplot(
        data=plot_df, x='Email Class', y=feature,
        hue='Email Class', legend=False, showfliers=False, ax=ax
    )
    ax.set_title(f'{feature} by Class')
plt.tight_layout()
plt.show()

In [ ]:
# 7. VISUALIZE FEATURES MOST CORRELATED WITH SPAM
correlations = (
    pd.concat([X, y.rename('spam')], axis=1)
    .corr(numeric_only=True)['spam']
    .drop('spam')
)
top_correlations = correlations.reindex(
    correlations.abs().sort_values(ascending=False).head(15).index
).sort_values()

plt.figure(figsize=(9, 6))
colors = ['#4C72B0' if value < 0 else '#DD8452' for value in top_correlations]
plt.barh(top_correlations.index, top_correlations.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 15 Feature Correlations with Spam')
plt.xlabel('Pearson Correlation with Target')
plt.tight_layout()
plt.show()

In [ ]:
# 8. VISUALIZE THE HIGH-DIMENSIONAL DATA IN TWO PRINCIPAL COMPONENTS
eda_scaler = StandardScaler()
X_eda_scaled = eda_scaler.fit_transform(X)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_eda_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Email Class'] = y.map({0: 'Non-spam', 1: 'Spam'})

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2', hue='Email Class',
    alpha=0.55, s=35
)
plt.title('Spambase Projected onto Two Principal Components')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.tight_layout()
plt.show()

In [ ]:
# 9. CREATE STRATIFIED TRAINING AND TEST SETS
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print('Training set:', X_train.shape)
print('Testing set:', X_test.shape)
print('\nTraining target proportions:\n', y_train.value_counts(normalize=True).sort_index().round(4))
print('\nTesting target proportions:\n', y_test.value_counts(normalize=True).sort_index().round(4))

## Linear SVM implementation

The model explicitly passes the string `'linear'` to `SVC`. A `StandardScaler` is placed before the classifier because SVM margins depend on feature scale.

In [ ]:
# 10. IMPLEMENT A BASELINE LINEAR SVM
baseline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear', C=1.0))
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_score = baseline_model.decision_function(X_test)

baseline_metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, baseline_pred),
    'Precision': precision_score(y_test, baseline_pred, zero_division=0),
    'Recall': recall_score(y_test, baseline_pred, zero_division=0),
    'F1-score': f1_score(y_test, baseline_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, baseline_score)
}).round(4)

print('Baseline Linear SVM Metrics:\n', baseline_metrics)
print('Support vectors:', baseline_model.named_steps['svc'].n_support_.sum())

## Hyperparameter tuning

The grid search tunes:

- `C`: controls the tradeoff between a wider margin and fewer training errors.
- `class_weight`: compares the original class weighting with automatic balancing.

The kernel remains fixed as `'linear'`. Scaling occurs inside each cross-validation fold to prevent preprocessing leakage.

In [ ]:
# 11. TUNE THE LINEAR SVM WITH GRID SEARCH
linear_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='linear'))
])

parameter_grid = {
    'svc__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'svc__class_weight': [None, 'balanced']
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid_search = GridSearchCV(
    estimator=linear_pipeline,
    param_grid=parameter_grid,
    scoring='f1',
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print('Best hyperparameters:', grid_search.best_params_)
print('Best mean cross-validation F1-score:', round(grid_search.best_score_, 4))

In [ ]:
# 12. DISPLAY AND VISUALIZE HYPERPARAMETER-TUNING RESULTS
cv_results = pd.DataFrame(grid_search.cv_results_)
tuning_table = (
    cv_results[[
        'rank_test_score', 'param_svc__C', 'param_svc__class_weight',
        'mean_train_score', 'mean_test_score', 'std_test_score'
    ]]
    .sort_values('rank_test_score')
    .reset_index(drop=True)
)
display(tuning_table.round(4))

plot_results = cv_results.copy()
plot_results['Class Weight'] = plot_results['param_svc__class_weight'].astype(str)

plt.figure(figsize=(9, 6))
for class_weight, group in plot_results.groupby('Class Weight'):
    group = group.sort_values('param_svc__C')
    plt.plot(
        group['param_svc__C'].astype(float), group['mean_test_score'],
        marker='o', linewidth=2, label=f'Validation: {class_weight}'
    )
    plt.plot(
        group['param_svc__C'].astype(float), group['mean_train_score'],
        marker='x', linestyle='--', alpha=0.75, label=f'Training: {class_weight}'
    )

plt.xscale('log')
plt.ylim(0.75, 1.01)
plt.xlabel('C (log scale)')
plt.ylabel('Mean F1-score')
plt.title('Linear SVM Hyperparameter-Tuning Results')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 13. EVALUATE THE TUNED LINEAR SVM ON UNSEEN TEST DATA
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_score = best_model.decision_function(X_test)

tuned_metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_score)
}).round(4)

metric_comparison = pd.DataFrame({
    'Baseline': baseline_metrics,
    'Tuned Linear SVM': tuned_metrics
})

display(metric_comparison)
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n')
print(classification_report(
    y_test, y_pred, target_names=['Non-spam', 'Spam'], digits=4
))

In [ ]:
# 14. VISUALIZE FINAL MODEL PERFORMANCE
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Non-spam', 'Spam'],
    cmap='Blues', colorbar=False, ax=axes[0]
)
axes[0].set_title('Tuned Linear SVM Confusion Matrix')

RocCurveDisplay.from_predictions(
    y_test, y_score, name='Linear SVM', ax=axes[1]
)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random classifier')
axes[1].set_title('ROC Curve')
axes[1].legend()

PrecisionRecallDisplay.from_predictions(
    y_test, y_score, name='Linear SVM', ax=axes[2]
)
axes[2].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

In [ ]:
# 15. VISUALIZE THE MOST INFLUENTIAL LINEAR SVM COEFFICIENTS
linear_svc = best_model.named_steps['svc']
coefficients = pd.Series(linear_svc.coef_[0], index=X.columns)
important_coefficients = coefficients.reindex(
    coefficients.abs().sort_values(ascending=False).head(20).index
).sort_values()

plt.figure(figsize=(10, 7))
coefficient_colors = [
    '#4C72B0' if value < 0 else '#DD8452'
    for value in important_coefficients
]
plt.barh(important_coefficients.index, important_coefficients.values,
         color=coefficient_colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 20 Standardized Linear SVM Coefficients')
plt.xlabel('Coefficient: Negative → Non-spam, Positive → Spam')
plt.tight_layout()
plt.show()

In [ ]:
# 16. VISUALIZE A LINEAR DECISION BOUNDARY IN TWO PCA DIMENSIONS
# This visualization-only model uses two PCs; the evaluated model above uses all 57 features.
pca_model = SVC(
    kernel='linear',
    C=grid_search.best_params_['svc__C'],
    class_weight=grid_search.best_params_['svc__class_weight']
)
pca_model.fit(X_pca, y)

x_min, x_max = np.percentile(X_pca[:, 0], [1, 99])
y_min, y_max = np.percentile(X_pca[:, 1], [1, 99])
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 350),
    np.linspace(y_min, y_max, 350)
)
decision_values = pca_model.decision_function(
    np.c_[xx.ravel(), yy.ravel()]
).reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, decision_values > 0, alpha=0.20, cmap='coolwarm')
plt.contour(
    xx, yy, decision_values, levels=[-1, 0, 1],
    colors=['gray', 'black', 'gray'], linestyles=['--', '-', '--']
)
plt.scatter(
    X_pca[:, 0], X_pca[:, 1], c=y, cmap='coolwarm',
    s=22, alpha=0.50, edgecolor='none'
)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.title('Linear SVM Boundary in Two PCA Dimensions')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.tight_layout()
plt.show()

In [ ]:
# 17. FINAL SUMMARY
print('Linear SVM tutorial completed.')
print('Kernel used: linear')
print('Best hyperparameters:', grid_search.best_params_)
print('Best cross-validation F1-score:', round(grid_search.best_score_, 4))
print('Unseen test F1-score:', tuned_metrics['F1-score'])
print('Unseen test ROC-AUC:', tuned_metrics['ROC-AUC'])
print('The test set was not used during hyperparameter selection.')